## Backtesting - Points XGBoost model

In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("could not find repo root (pyproject.toml)")


ROOT = repo_root()
for folder in (ROOT, ROOT / "notebooks/nba/points/src"):
    folder_text = str(folder)
    if folder_text not in sys.path:
        sys.path.insert(0, folder_text)

from models.shared.artifacts import (
    load_model_bundle,
    predict_calibrated_quantiles,
)
from models.shared.points_calibration import quantile_line_probabilities
from points_xgb.data import load_regular_season

MODEL_PATH = ROOT / "models/saved_models/pts_nba_model_2026-11-02.joblib"
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"missing production model: {MODEL_PATH}")
bundle = load_model_bundle(MODEL_PATH)
PTS_FEATURES = list(bundle["feature_names"])
print(
    f"Model {MODEL_PATH.name}: "
    f"trained through {pd.Timestamp(bundle['train_end']).date()}, "
    f"holdout through {pd.Timestamp(bundle['val_end']).date()}"
)
print(f"Features ({len(PTS_FEATURES)}): {', '.join(PTS_FEATURES)}")

# PIT map stored on the bundle. Knots are the calibrated quantiles.
if bundle.get("calibration", {}).get("method") != "pit_cdf":
    raise KeyError("bundle has no pit_cdf calibration")
TAUS = np.asarray(bundle["calibration"]["taus"], dtype=float)
QUANTILE_KEYS = tuple(f"q_{tau:.2f}" for tau in TAUS)
if "q_0.50" not in QUANTILE_KEYS:
    raise KeyError("bundle has no q_0.50")
print(
    "Probabilities use the calibrated quantile grid, "
    f"taus {', '.join(f'{tau:.2f}' for tau in TAUS)}."
)


def quantile_knots(frame: pd.DataFrame) -> np.ndarray:
    """Calibrated quantile grid for rows that already have the model features."""
    knots = predict_calibrated_quantiles(bundle, frame)
    return np.column_stack([knots[key].to_numpy(dtype=float) for key in QUANTILE_KEYS])


Loaded /Users/alexgonzalez/Documents/nba_quant/models/saved_models/pts_nba_model_2026-11-02.joblib
Model pts_nba_model_2026-11-02.joblib: trained through 2026-04-12, holdout through 2026-11-02
Features (26): pts_mean_10, pts_std_10, pts_lag_1, pts_trend, pts_rank_10, two_pa_rolling_10, two_pct_smoothed, fg3a_rolling_10, three_pct_smoothed, fta_rolling_10, ft_pct_smoothed, min_mean_10, min_mean_3_minus_10, min_rank_10, min_sd_10, ts_x_usg, usg_rolling_10, usg_trend, usg_rank_10, starting, starter_rank, start_rate_10, opp_def_rating_mean_10, expected_pace, implied_team_total, player_team_spread
Probabilities are the raw quantile CDF, taus 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95.


In [4]:
import importlib

from src.features.points.closing import load_closing_points, match_player_ids
from src.pipeline.silver.rotowire import normalize_team_abbreviation

# The module name starts with a digit, so a plain import statement cannot load it.
live_points = importlib.import_module("src.live.01_predict_points")
FEATURES = live_points.FEATURES
build_points_live = live_points.build_points_live
lineups_on = live_points.lineups_on
pinnacle_games_on = live_points.pinnacle_games_on

if list(FEATURES) != list(PTS_FEATURES):
    raise AssertionError("live feature list does not match PTS_FEATURES")

# Only these Pacific slate dates are featured / scored.
SLATE_DATES = ("2026-10-25","2026-10-26","2026-10-27","2026-10-28","2026-10-30","2026-10-31", "2026-11-01", "2026-11-02", "2026-11-03")

raw = load_regular_season(ROOT)
lines = load_closing_points(ROOT / "data/props/prizepicks/nba/closing")
lines = match_player_ids(lines, raw)
lines["slate_date"] = pd.to_datetime(lines["slate_date"]).dt.normalize()
wanted = {pd.Timestamp(day).normalize() for day in SLATE_DATES}
lines = lines.loc[lines["slate_date"].isin(wanted)].copy()
if lines.empty:
    raise ValueError(f"No PrizePicks closing lines for {sorted(SLATE_DATES)}")

unmatched = sorted(lines.loc[lines["player_id"].isna(), "player_name"].unique())
print(f"Standard points lines on {', '.join(SLATE_DATES)}: {len(lines):,}")
print(
    f"Unmatched names ({len(unmatched)}): "
    f"{', '.join(unmatched) if unmatched else '(none)'}"
)

name_by_id = (
    raw.sort_values(["game_date", "game_id"], kind="mergesort")
    .groupby("player_id", sort=False)["player_name"]
    .last()
)
raw_dates = pd.to_datetime(raw["game_date"]).dt.normalize()
parts = []
modeled_parts = []
for slate, group in lines.dropna(subset=["player_id"]).groupby("slate_date", sort=True):
    slate = pd.Timestamp(slate).normalize()
    day = slate.date().isoformat()
    before = raw_dates < slate
    same_day = raw_dates.eq(slate)
    print(
        f"{day}: history rows {int(before.sum()):,}, "
        f"{int(same_day.sum()):,} same-day log rows held out of features"
    )
    history = raw.loc[before]
    games = pinnacle_games_on(day)
    lineups = lineups_on(day)
    if not games:
        print(f"  no Pinnacle team file for {day}; spread and total stay missing")
    if not lineups:
        print(f"  no RotoWire file for {day}; starter flags stay missing")
    print(f"  pinnacle games {len(games):,}, rotowire teams {len(lineups):,}")

    player_ids = group["player_id"].drop_duplicates().tolist()
    names = []
    missing_history = []
    id_by_name = {}
    for player_id in player_ids:
        name = name_by_id.get(player_id)
        if name is None or not history["player_name"].eq(name).any():
            missing_history.append(str(name or player_id))
            continue
        names.append(name)
        id_by_name[name] = player_id
    if missing_history:
        print(f"  no games before {day}: {', '.join(missing_history)}")
    if not names:
        raise ValueError(f"no feature rows for {day}")

    # One batched live build per slate; starting from RotoWire posted fives.
    live = build_points_live(
        history,
        names,
        day,
        games=games,
        lineups=lineups,
        extras=(),
    )
    rows = []
    for name, row in live.iterrows():
        player_id = id_by_name[name]
        prior = history.loc[history["player_name"].eq(name)].sort_values(
            ["game_date", "game_id"], kind="mergesort"
        )
        rows.append(
            {
                "player_id": player_id,
                "player_name": name,
                "team_abbreviation": normalize_team_abbreviation(
                    prior["team_abbreviation"].iloc[-1]
                ),
                "history_through": pd.to_datetime(prior["game_date"].iloc[-1]),
                **row.to_dict(),
            }
        )
    featured = pd.DataFrame(rows)
    if featured.empty:
        raise ValueError(f"no feature rows for {day}")

    if not lineups:
        missing_five = featured.index
    else:
        missing_five = featured.index[
            ~featured["team_abbreviation"].isin(lineups)
        ]
    if len(missing_five):
        featured.loc[missing_five, "starting"] = np.nan
        print(f"  {len(missing_five):,} players had no posted five")

    through = pd.to_datetime(featured["history_through"]).dt.normalize()
    if not (through < slate).all():
        raise AssertionError(f"{day} used a game on or after the closing date")
    print(
        f"  history through {through.max().date()}, "
        f"{len(featured):,} players featured"
    )
    played = (
        raw.loc[same_day, ["player_id", "pts"]]
        .drop_duplicates("player_id")
        .rename(columns={"pts": "pts_played"})
    )
    overlap = [
        column
        for column in featured.columns
        if column in group.columns and column != "player_id"
    ]
    board = featured.loc[:, ["player_id", "history_through", *PTS_FEATURES]].copy()
    board["slate_date"] = pd.Timestamp(slate).normalize()
    board = board.merge(played, on="player_id", how="left")
    modeled_parts.append(board)
    parts.append(
        group.merge(featured.drop(columns=overlap), on="player_id", how="left").merge(
            played, on="player_id", how="left"
        )
    )

closing = pd.concat(parts, ignore_index=True)
missing = [column for column in PTS_FEATURES if column not in closing.columns]
if missing:
    raise KeyError(f"Closing frame is missing features: {missing}")

closing[PTS_FEATURES] = closing[PTS_FEATURES].apply(pd.to_numeric, errors="coerce")
ready = closing["pts_lag_1"].notna().to_numpy()
mu = np.full(len(closing), np.nan)
p_over = np.full(len(closing), np.nan)
if ready.any():
    grid = quantile_knots(closing.loc[ready, PTS_FEATURES])
    mu[ready] = grid[:, QUANTILE_KEYS.index("q_0.50")]
    p_over[ready], _push = quantile_line_probabilities(
        closing.loc[ready, "line"].to_numpy(), grid, TAUS
    )
unscored = int((~ready).sum())
if unscored:
    print(f"{unscored:,} lines had no prior games and were not scored")
closing["mu"] = mu
closing["p_over"] = p_over
closing["went_over"] = closing["pts_played"].gt(closing["line"])
modeled = pd.concat(modeled_parts, ignore_index=True)
modeled[PTS_FEATURES] = modeled[PTS_FEATURES].apply(pd.to_numeric, errors="coerce")
modeled_ready = modeled["pts_lag_1"].notna()
modeled["mu"] = np.nan
for key in QUANTILE_KEYS:
    modeled[key] = np.nan
if modeled_ready.any():
    grid = quantile_knots(modeled.loc[modeled_ready, PTS_FEATURES])
    modeled.loc[modeled_ready, list(QUANTILE_KEYS)] = grid
    modeled.loc[modeled_ready, "mu"] = grid[:, QUANTILE_KEYS.index("q_0.50")]
print(
    f"Priced {closing['p_over'].notna().sum():,} PrizePicks lines on "
    f"{', '.join(SLATE_DATES)}. "
    f"Quantile grids saved for {int(modeled_ready.sum()):,} player-slates."
)
closing[
    [
        "slate_date",
        "player_name",
        "line",
        "mu",
        "p_over",
        "pts_played",
        "history_through",
    ]
].sort_values(["slate_date", "p_over"], ascending=[True, False]).head(20)


Standard points lines on 2026-10-25, 2026-10-26, 2026-10-27, 2026-10-28, 2026-10-30, 2026-10-31, 2026-11-01, 2026-11-02, 2026-11-03: 762
Unmatched names (0): (none)
2026-10-25: history rows 177,550, 157 same-day log rows held out of features
  pinnacle games 236, rotowire teams 14


/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:187: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result["usg_rolling_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:204: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{name}_min_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:211: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

  1 players had no posted five
  history through 2026-10-24, 67 players featured
2026-10-26: history rows 177,707, 212 same-day log rows held out of features
  pinnacle games 147, rotowire teams 18


/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:187: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result["usg_rolling_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:204: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{name}_min_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:211: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

  history through 2026-10-25, 119 players featured
2026-10-27: history rows 177,919, 94 same-day log rows held out of features
  pinnacle games 87, rotowire teams 8


/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:187: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result["usg_rolling_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:204: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{name}_min_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:211: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

... (8 outputs left out)

  history through 2026-10-30, 42 players featured
2026-11-02: history rows 178,817, 334 same-day log rows held out of features
  pinnacle games 384, rotowire teams 30


/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:187: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result["usg_rolling_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:204: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{name}_min_10"] = prior_roll(
/Users/alexgonzalez/Documents/nba_quant/src/features/points/scoring.py:211: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

  history through 2026-11-01, 146 players featured
1 lines had no prior games and were not scored
Priced 761 PrizePicks lines on 2026-10-25, 2026-10-26, 2026-10-27, 2026-10-28, 2026-10-30, 2026-10-31, 2026-11-01, 2026-11-02, 2026-11-03. Quantile grids saved for 759 player-slates.


,slate_date,player_name,line,mu,p_over,pts_played,history_through
66,2026-10-25,Ziaire Williams,8.0,13.369611,0.806027,11.0,2026-10-23
0,2026-10-25,Aaron Nesmith,9.5,14.466984,0.778876,13.0,2026-10-23
21,2026-10-25,Isaiah Collier,6.5,10.218130,0.778840,6.0,2026-10-23
47,2026-10-25,Matisse Thybulle,5.5,8.419417,0.758922,11.0,2026-10-23
31,2026-10-25,Jaren Jackson,17.5,22.221119,0.751036,20.0,2026-10-23
33,2026-10-25,Jaxson Hayes,6.5,9.048096,0.723253,9.0,2026-10-23
12,2026-10-25,Darius Acuff Jr.,20.0,25.498455,0.719653,22.0,2026-10-23
65,2026-10-25,Wendell Carter,11.5,14.530742,0.717451,9.0,2026-10-23
4,2026-10-25,Andrew Nembhard,13.5,16.238224,0.677589,14.0,2026-10-23
56,2026-10-25,Rudy Gobert,8.5,10.033257,0.656417,8.0,2026-10-23


In [5]:
import json
from pathlib import Path

from src.features.points.closing import match_player_ids

POINTS_STATS = {"points", "pts"}
BOOK_CLOSING = {
    "prizepicks": ROOT / "data/props/prizepicks/nba/closing",
    "underdog": ROOT / "data/props/underdogs/nba/closing",
    "novig": ROOT / "data/props/novig/nba/closing",
    "prophetx": ROOT / "data/props/prophetx/nba/closing",
    "pinnacle": ROOT / "data/props/pinnacle/nba/closing",
}


def _american_profit(price):
    """Profit on a $1 stake. Negative American odds risk more than they pay."""
    price = np.asarray(price, dtype=float)
    return np.where(price < 0, 100.0 / np.abs(price), price / 100.0)


def _implied(price):
    price = price.astype(float)
    negative = price < 0
    out = pd.Series(np.nan, index=price.index, dtype=float)
    out.loc[negative] = price.loc[negative].abs() / (price.loc[negative].abs() + 100)
    out.loc[~negative] = 100 / (price.loc[~negative] + 100)
    return out


def _load_points_sides(book, directory):
    rows = []
    folder = Path(directory)
    for path in sorted(folder.glob("*_nba_closing_*.json")):
        payload = json.loads(path.read_text())
        slate = pd.Timestamp(payload["date"]).normalize()
        for player in payload.get("players", []):
            for offer in player.get("lines", []):
                stat = str(offer.get("stat_type") or "").strip().lower()
                if stat not in POINTS_STATS:
                    continue
                if book == "prizepicks" and offer.get("odds_type") != "standard":
                    continue
                rows.append(
                    {
                        "slate_date": slate,
                        "book": book,
                        "player_name": player.get("player"),
                        "line": float(offer["line_score"]),
                        "side": offer.get("side"),
                        "american_price": offer.get("american_price"),
                    }
                )
    return pd.DataFrame(rows)


def _main_points_line(sides):
    """One two-way points line per player.

    Novig and ProphetX keep alternate ladders. The main line is the
    rung where over and under are closest to each other. PrizePicks
    standard is one line, priced -119 on both sides.
    """
    prizepicks = sides.loc[sides["book"].eq("prizepicks")].copy()
    prizepicks = prizepicks.drop_duplicates(["slate_date", "player_name", "line"])
    prizepicks["over_american"] = -119
    prizepicks["under_american"] = -119

    other = sides.loc[~sides["book"].eq("prizepicks")].dropna(
        subset=["side", "american_price"]
    )
    other = other.assign(side=other["side"].astype("string").str.lower())
    if other.empty:
        wide = pd.DataFrame(
            columns=[
                "slate_date",
                "book",
                "player_name",
                "line",
                "over",
                "under",
            ]
        )
        one_sided = 0
    else:
        wide = (
            other.pivot_table(
                index=["slate_date", "book", "player_name", "line"],
                columns="side",
                values="american_price",
                aggfunc="first",
            )
            .reset_index()
        )
        for column in ("over", "under"):
            if column not in wide.columns:
                wide[column] = np.nan
        paired = wide["over"].notna() & wide["under"].notna()
        one_sided = int((~paired).sum())
        wide = wide.loc[paired].copy()
        wide["gap"] = (_implied(wide["over"]) - _implied(wide["under"])).abs()
        wide = wide.sort_values("gap").drop_duplicates(
            ["slate_date", "book", "player_name"]
        )
    wide = wide.rename(columns={"over": "over_american", "under": "under_american"})
    keep = [
        "slate_date",
        "book",
        "player_name",
        "line",
        "over_american",
        "under_american",
    ]
    markets = pd.concat([prizepicks[keep], wide[keep]], ignore_index=True)
    return markets, one_sided


sides = pd.concat(
    [_load_points_sides(book, folder) for book, folder in BOOK_CLOSING.items()],
    ignore_index=True,
)
sides["slate_date"] = pd.to_datetime(sides["slate_date"]).dt.normalize()
# Reuse SLATE_DATES from the feature cell (defaults to 2026-10-30).
_slate_dates = globals().get("SLATE_DATES", ("2026-10-30",))
_wanted = {pd.Timestamp(day).normalize() for day in _slate_dates}
sides = sides.loc[sides["slate_date"].isin(_wanted)].copy()
if sides.empty:
    raise ValueError(f"No closing sides for {_slate_dates}")
print(
    f"Loaded {len(sides):,} point sides on "
    f"{', '.join(str(pd.Timestamp(d).date()) for d in sorted(_wanted))} "
    f"across {sides['book'].nunique()} books"
)
markets, one_sided = _main_points_line(sides)
markets = match_player_ids(markets, raw)
unmatched = sorted(markets.loc[markets["player_id"].isna(), "player_name"].unique())

means = modeled if "modeled" in globals() else closing
means = means.copy()
if "slate_date" not in means.columns and "slate_date_x" in means.columns:
    means = means.rename(
        columns={"slate_date_x": "slate_date", "player_name_x": "player_name"}
    )
means["slate_date"] = pd.to_datetime(means["slate_date"]).dt.normalize()
means = means.loc[:, ["slate_date", "player_id", "mu", "pts_played", *QUANTILE_KEYS]].drop_duplicates(
    ["slate_date", "player_id"]
)
book = markets.merge(means, on=["slate_date", "player_id"], how="left")
finite = np.isfinite(book.loc[:, list(QUANTILE_KEYS)].to_numpy(dtype=float)).all(axis=1)
missing_mu = int((~finite).sum())
book = book.loc[finite].reset_index(drop=True)

grid = book.loc[:, list(QUANTILE_KEYS)].to_numpy(dtype=float)
p_over, p_push = quantile_line_probabilities(book["line"].to_numpy(), grid, TAUS)
p_under = 1.0 - p_over - p_push
over_profit = _american_profit(book["over_american"])
under_profit = _american_profit(book["under_american"])
book["p_over"] = p_over
book["p_under"] = p_under
book["ev_over"] = p_over * over_profit - p_under
book["ev_under"] = p_under * under_profit - p_over
take_over = book["ev_over"] >= book["ev_under"]
book["side"] = np.where(take_over, "over", "under")
book["p_side"] = np.where(take_over, p_over, p_under)
book["price"] = np.where(take_over, book["over_american"], book["under_american"])
book["ev"] = np.maximum(book["ev_over"], book["ev_under"])
book["win_profit"] = np.where(take_over, over_profit, under_profit)

settled = book["pts_played"].notna()
won = np.where(
    take_over,
    book["pts_played"].gt(book["line"]),
    book["pts_played"].lt(book["line"]),
)
pushed = book["pts_played"].eq(book["line"])
book["result"] = np.where(
    ~settled,
    "pending",
    np.where(pushed, "push", np.where(won, "win", "loss")),
)
book["profit"] = np.where(
    book["result"].eq("win"),
    book["win_profit"],
    np.where(book["result"].eq("loss"), -1.0, 0.0),
)

graded = book.loc[book["result"].isin(["win", "loss"])].copy()
plus = book.loc[book["ev"].gt(0)].copy()
plus_graded = plus.loc[plus["result"].isin(["win", "loss"])]
print(
    f"Main points closes: {len(markets):,}. "
    f"One-sided offers left out: {one_sided:,}. "
    f"Unmatched names ({len(unmatched)}): "
    f"{', '.join(unmatched) if unmatched else '(none)'}"
)
print(
    f"Scored {len(book):,} lines with a quantile CDF. "
    f"{missing_mu:,} had no distribution."
)
print(
    f"Settled {int(settled.sum()):,}. "
    f"Pushes {int((book['result'] == 'push').sum()):,}. "
    f"+EV {len(plus):,} ({int(plus['result'].ne('pending').sum()):,} settled)."
)
if len(graded):
    print(
        f"Favored side hit {graded['result'].eq('win').mean():.1%} "
        f"on {len(graded):,} decided lines "
        f"(mean probability {graded['p_side'].mean():.1%})."
    )
if len(plus_graded):
    plus_settled = plus.loc[plus["result"].ne("pending")]
    print(
        f"+EV hit {plus_graded['result'].eq('win').mean():.1%} "
        f"on {len(plus_graded):,} decided bets. "
        f"ROI {plus_settled['profit'].sum() / len(plus_settled):.1%} per $1 stake."
    )

by_book = (
    book.loc[book["result"].ne("pending")]
    .groupby("book", observed=False)
    .apply(
        lambda frame: pd.Series(
            {
                "n": len(frame),
                "decided": int(frame["result"].isin(["win", "loss"]).sum()),
                "hit_rate": frame.loc[
                    frame["result"].isin(["win", "loss"]), "result"
                ]
                .eq("win")
                .mean(),
                "roi": frame["profit"].sum() / len(frame),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
print("Settled closes by book. ROI is profit per $1 at that book's price.")
display(by_book)

# Calibration: when the model says X%, do those bets hit about X%?
prob_edges = [0.50, 0.55, 0.60, 0.65, 0.70, 1.01]
prob_labels = ["50\u201355%", "55\u201360%", "60\u201365%", "65\u201370%", "70%+"]
graded["bucket"] = pd.cut(
    graded["p_side"],
    bins=prob_edges,
    labels=prob_labels,
    right=False,
    include_lowest=True,
)
accuracy = (
    graded.groupby("bucket", observed=False)
    .agg(
        n=("result", "size"),
        wins=("result", lambda series: int(series.eq("win").sum())),
        actual_hit_rate=("result", lambda series: series.eq("win").mean()),
        predicted_prob=("p_side", "mean"),
        roi=("profit", "mean"),
    )
    .reset_index()
)
accuracy["calibration_gap"] = (
    accuracy["actual_hit_rate"] - accuracy["predicted_prob"]
)
print(
    "Favored-side calibration by predicted win probability. "
    "predicted_prob is the bucket's mean model probability; "
    "actual_hit_rate is how often that side won; "
    "calibration_gap = actual - predicted (near 0 is well calibrated)."
)
display(accuracy.round(3))

# Same check on raw P(over), not just the favored side.
over_decided = book.loc[
    book["pts_played"].notna() & book["p_over"].notna()
].copy()
over_decided["over_hit"] = over_decided["pts_played"].gt(over_decided["line"])
# Push rows (exact line) are neither over hits nor misses for this table.
over_decided = over_decided.loc[
    ~over_decided["pts_played"].eq(over_decided["line"])
].copy()
over_edges = [0.0, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 1.01]
over_labels = [
    "<40%",
    "40\u201345%",
    "45\u201350%",
    "50\u201355%",
    "55\u201360%",
    "60\u201365%",
    "65%+",
]
over_decided["p_over_bucket"] = pd.cut(
    over_decided["p_over"],
    bins=over_edges,
    labels=over_labels,
    right=False,
    include_lowest=True,
)
over_calibration = (
    over_decided.groupby("p_over_bucket", observed=False)
    .agg(
        n=("over_hit", "size"),
        actual_over_rate=("over_hit", "mean"),
        predicted_p_over=("p_over", "mean"),
    )
    .reset_index()
)
over_calibration["calibration_gap"] = (
    over_calibration["actual_over_rate"]
    - over_calibration["predicted_p_over"]
)
print(
    "Over-side calibration: buckets of model P(over) vs how often the over hit. "
    "Pushes excluded."
)
display(over_calibration.round(3))

ev_edges = [-np.inf, 0.0, 0.05, 0.10, 0.20, np.inf]
ev_labels = ["<0%", "0–5%", "5–10%", "10–20%", "20%+"]
graded["ev_bucket"] = pd.cut(
    graded["ev"], bins=ev_edges, labels=ev_labels, right=False
)
ev_accuracy = (
    graded.groupby("ev_bucket", observed=False)
    .agg(
        n=("result", "size"),
        wins=("result", lambda series: int(series.eq("win").sum())),
        hit_rate=("result", lambda series: series.eq("win").mean()),
        mean_ev=("ev", "mean"),
        mean_probability=("p_side", "mean"),
        roi=("profit", "mean"),
    )
    .reset_index()
)
print("Favored-side results by expected value per $1 stake.")
display(ev_accuracy)

top = (
    book.loc[book["ev"].gt(0)]
    .sort_values("ev", ascending=False)
    .loc[
        :,
        [
            "slate_date",
            "book",
            "player_name",
            "side",
            "line",
            "price",
            "mu",
            "p_side",
            "ev",
            "pts_played",
            "result",
        ],
    ]
    .head(20)
)
print("Highest expected value per $1 stake, at each book's closing price.")
display(top)


Loaded 25,926 point sides on 2026-10-25, 2026-10-26, 2026-10-27, 2026-10-28, 2026-10-30, 2026-10-31, 2026-11-01, 2026-11-02, 2026-11-03 across 5 books
Main points closes: 3,813. One-sided offers left out: 11,102. Unmatched names (6): Alexandre Sarr, Cam Johnson, Deuce McBride, Gregory Jackson, Herb Jones, Lu Dort
Scored 3,257 lines with a quantile CDF. 556 had no distribution.
Settled 3,147. Pushes 12. +EV 2,602 (2,515 settled).
Favored side hit 49.0% on 3,135 decided lines (mean probability 59.6%).
+EV hit 49.3% on 2,504 decided bets. ROI -6.3% per $1 stake.
Settled closes by book. ROI is profit per $1 at that book's price.


,book,n,decided,hit_rate,roi
0,novig,630.0,630.0,0.490476,-0.083796
1,pinnacle,579.0,579.0,0.481865,-0.095446
2,prizepicks,733.0,721.0,0.490985,-0.094844
3,prophetx,584.0,584.0,0.501712,-0.057113
4,underdog,621.0,621.0,0.486312,-0.071416


Favored-side calibration by predicted win probability. predicted_prob is the bucket's mean model probability; actual_hit_rate is how often that side won; calibration_gap = actual - predicted (near 0 is well calibrated).


,bucket,n,wins,actual_hit_rate,predicted_prob,roi,calibration_gap
0,50–55%,835,393,0.471,0.527,-0.103,-0.056
1,55–60%,826,405,0.490,0.574,-0.076,-0.084
2,60–65%,630,295,0.468,0.623,-0.128,-0.155
3,65–70%,371,198,0.534,0.673,-0.014,-0.139
4,70%+,309,177,0.573,0.761,0.001,-0.188


Over-side calibration: buckets of model P(over) vs how often the over hit. Pushes excluded.


,p_over_bucket,n,actual_over_rate,predicted_p_over,calibration_gap
0,<40%,739,0.455,0.329,0.126
1,40–45%,393,0.471,0.426,0.045
2,45–50%,486,0.447,0.475,-0.029
3,50–55%,477,0.423,0.526,-0.102
4,55–60%,446,0.453,0.574,-0.121
5,60–65%,288,0.392,0.625,-0.232
6,65%+,306,0.542,0.710,-0.167


Favored-side results by expected value per $1 stake.


,ev_bucket,n,wins,hit_rate,mean_ev,mean_probability,roi
0,<0%,631,303,0.480190,-0.047363,0.535805,-0.152261
1,0–5%,484,236,0.487603,0.025302,0.544930,-0.085462
2,5–10%,466,225,0.482833,0.075147,0.568119,-0.086341
3,10–20%,747,366,0.489960,0.147632,0.604670,-0.064888
4,20%+,807,407,0.504337,0.323292,0.683043,-0.036099


Highest expected value per $1 stake, at each book's closing price.


,slate_date,book,player_name,side,line,price,mu,p_side,ev,pts_played,result
3254,2026-10-26,novig,Klay Thompson,over,9.5,426,12.358397,0.678734,2.570140,6.0,loss
3224,2026-10-26,novig,Devin Booker,over,14.5,174,29.038986,0.950000,1.603000,3.0,loss
3253,2026-10-25,prophetx,Jaxson Hayes,over,9.5,355,9.048096,0.505128,1.298334,9.0,loss
3208,2026-10-26,novig,Ajay Mitchell,under,24.5,120,12.482292,0.947720,1.084983,30.0,loss
3242,2026-10-26,novig,Max Christie,over,9.5,223,10.695134,0.626888,1.024848,11.0,win
1490,2026-11-02,underdog,Kobe Brown,under,12.5,-106,2.630255,0.950000,0.846226,16.0,loss
707,2026-11-02,prizepicks,Kobe Brown,under,12.5,-119,2.630255,0.950000,0.748319,16.0,loss
2969,2026-10-26,prophetx,Bilal Coulibaly,over,5.5,-104,11.398363,0.884474,0.734930,3.0,loss
1972,2026-10-25,prophetx,Ausar Thompson,under,10.5,104,5.822257,0.850108,0.734221,NaN,pending
2303,2026-11-02,prophetx,Jarace Walker,under,18.5,-112,9.296859,0.914652,0.731305,15.0,win


In [6]:
line_edges = [-np.inf, 10, 15, 20, 25, np.inf]
line_labels = ["<10", "10–15", "15–20", "20–25", "25+"]
by_line = graded.copy()
by_line["line_bucket"] = pd.cut(
    by_line["line"], bins=line_edges, labels=line_labels, right=False
)
line_accuracy = (
    by_line.groupby("line_bucket", observed=False)
    .agg(
        n=("result", "size"),
        wins=("result", lambda series: int(series.eq("win").sum())),
        hit_rate=("result", lambda series: series.eq("win").mean()),
        mean_line=("line", "mean"),
        mean_probability=("p_side", "mean"),
        mean_ev=("ev", "mean"),
        roi=("profit", "mean"),
    )
    .reset_index()
)
print(
    "Favored side on decided closes, by the posted points line. "
    "A line of 10 is in 10–15, and 25 is in 25+."
)
display(line_accuracy)


Favored side on decided closes, by the posted points line. A line of 10 is in 10–15, and 25 is in 25+.


,line_bucket,n,wins,hit_rate,mean_line,mean_probability,mean_ev,roi
0,<10,1008,482,0.478175,7.978175,0.591418,0.116827,-0.105375
1,10–15,1010,503,0.498020,12.385644,0.591111,0.111971,-0.072336
2,15–20,587,282,0.480409,17.357751,0.609846,0.147469,-0.094255
3,20–25,398,183,0.459799,22.405779,0.599210,0.134271,-0.132157
4,25+,132,87,0.659091,28.045455,0.604931,0.134089,0.241856


In [ ]:
# One bankroll, walked slate by slate. Each night the card is the
# best price per player, then the TOP_N highest model EVs. Stakes are
# the Kelly fraction of that night's opening bankroll. Bets on the
# same slate are simultaneous, so the fractions are scaled down when
# they would ask for more than the whole bankroll.
BANKROLL = 1_000.0
TOP_N = 5
KELLY_FRACTION = 1.0
MIN_EV = 0.0

def _kelly_book(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """One bankroll for a single book's settled closes."""
    candidates = frame.loc[frame["result"].isin(["win", "loss", "push"])].copy()
    if candidates.empty:
        return pd.DataFrame(), pd.DataFrame(), {}
    candidates["p_push"] = (
        1.0 - candidates["p_over"] - candidates["p_under"]
    ).clip(lower=0.0)
    candidates["p_lose"] = (
        1.0 - candidates["p_side"] - candidates["p_push"]
    ).clip(lower=0.0)
    odds = candidates["win_profit"].to_numpy(dtype=float)
    p_win = candidates["p_side"].to_numpy(dtype=float)
    p_lose = candidates["p_lose"].to_numpy(dtype=float)
    denominator = odds * (p_win + p_lose)
    candidates["kelly"] = np.clip(
        np.divide(
            p_win * odds - p_lose,
            denominator,
            out=np.zeros_like(denominator),
            where=denominator > 0,
        ),
        0.0,
        1.0,
    )
    candidates = (
        candidates.sort_values(["ev", "kelly"], ascending=False)
        .drop_duplicates(["slate_date", "player_id"])
    )
    candidates = candidates.loc[
        candidates["ev"].gt(MIN_EV) & candidates["kelly"].gt(0)
    ]

    bankroll = float(BANKROLL)
    peak = bankroll
    max_drawdown = 0.0
    nights = []
    bet_log = []
    for slate, night in candidates.groupby("slate_date", sort=True):
        if bankroll <= 0:
            break
        picks = night.nlargest(TOP_N, "ev")
        opening = bankroll
        fractions = picks["kelly"].to_numpy(dtype=float) * KELLY_FRACTION
        if fractions.sum() > 1.0:
            fractions = fractions / fractions.sum()
        stakes = fractions * opening
        night_profit = 0.0
        for stake, (_, bet) in zip(stakes, picks.iterrows()):
            if bet["result"] == "win":
                pnl = stake * float(bet["win_profit"])
            elif bet["result"] == "loss":
                pnl = -stake
            else:
                pnl = 0.0
            night_profit += pnl
            bet_log.append(
                {
                    "book": bet["book"],
                    "slate_date": pd.Timestamp(slate).date(),
                    "player_name": bet["player_name"],
                    "side": bet["side"],
                    "line": bet["line"],
                    "price": bet["price"],
                    "ev": bet["ev"],
                    "kelly": bet["kelly"] * KELLY_FRACTION,
                    "stake": stake,
                    "result": bet["result"],
                    "pnl": pnl,
                }
            )
        bankroll = opening + night_profit
        peak = max(peak, bankroll)
        if peak > 0:
            max_drawdown = max(max_drawdown, (peak - bankroll) / peak)
        nights.append(
            {
                "book": picks["book"].iloc[0],
                "slate_date": pd.Timestamp(slate).date(),
                "bets": len(picks),
                "staked": float(stakes.sum()),
                "profit": night_profit,
                "bankroll": bankroll,
            }
        )

    path = pd.DataFrame(nights)
    log = pd.DataFrame(bet_log)
    flat_roi = (
        log["pnl"].sum() / log["stake"].sum() if len(log) and log["stake"].sum() else np.nan
    )
    summary = {
        "book": frame["book"].iloc[0],
        "bets": len(log),
        "hit_rate": log["result"].eq("win").mean() if len(log) else np.nan,
        "end_bankroll": bankroll,
        "profit": bankroll - BANKROLL,
        "roi": (bankroll - BANKROLL) / BANKROLL,
        "max_drawdown": max_drawdown,
        "flat_roi": flat_roi,
    }
    return path, log, summary


paths = []
logs = []
summaries = []
for book_name, frame in book.groupby("book", sort=True):
    path, log, summary = _kelly_book(frame)
    if summary:
        summaries.append(summary)
    if len(path):
        paths.append(path)
    if len(log):
        logs.append(log)

summary = pd.DataFrame(summaries).sort_values("roi", ascending=False)
path = pd.concat(paths, ignore_index=True) if paths else pd.DataFrame()
log = pd.concat(logs, ignore_index=True) if logs else pd.DataFrame()
print(
    f"Start ${BANKROLL:,.0f} in each book. Top {TOP_N} +EV bets per slate, "
    f"Kelly fraction {KELLY_FRACTION:g}, min EV {MIN_EV:g}."
)
display(summary)
display(path)
display(
    log.sort_values(["book", "slate_date", "ev"], ascending=[True, True, False])
    .reset_index(drop=True)
)
